# FL vs CL on PAD-UFES-20 — Kaggle runner

**Every session:** set `PLAN` below, then **Save Version → Save & Run All (Commit)**.
Settings: Accelerator **GPU T4 x2**, Internet **On** (first session downloads ImageNet weights).
Inputs: the PAD-UFES-20 dataset, the `fydp-fl-skin-code` dataset, and **this notebook's own previous output**
(so checkpoints carry over). Each run saves a checkpoint every epoch/round to `/kaggle/working/runs/`.
The runner stops starting new work after `HOURS`, so Kaggle always has time to save the outputs.

In [ ]:
# 1. Settings — change PLAN per session
PLAN = "calib"            # calib | week1a | week1b | week2 | ablations | ablations5
SPLITS = "splits_k3.csv"  # splits_k5.csv only for the ablations5 plan
HOURS = 11.0              # stop starting new work after this many hours (Kaggle limit: 12)

In [ ]:
# 2. Locate code and data
import glob, os, sys, shutil, subprocess, time, json
START = time.time()
cands = [p for p in glob.glob("/kaggle/input/**/fl_skin", recursive=True)
         if os.path.isdir(p) and "code_snapshot" not in p]
assert cands, "attach the fydp-fl-skin-code dataset"
CODE = os.path.dirname(cands[0])
DATA = os.path.dirname(glob.glob("/kaggle/input/**/metadata.csv", recursive=True)[0])
SPLITS_PATH = glob.glob(f"{CODE}/**/{SPLITS}", recursive=True)[0]
PLAN_PATH = f"{CODE}/plans/{PLAN}.json"
shutil.copytree(CODE, "/kaggle/working/code_snapshot", dirs_exist_ok=True)  # record of the exact code used
sys.path.insert(0, CODE)
print("code:", CODE, "\ndata:", DATA, "\nsplits:", SPLITS_PATH, "\nplan:", PLAN_PATH)
print(len(json.load(open(PLAN_PATH))), "runs in plan")

In [ ]:
# 3. Bring back checkpoints from the previous session's output
from fl_skin.restore import restore
r = restore("/kaggle/input", "/kaggle/working")
print(f"restored {len(r)} run folders")

In [ ]:
# 4. GPUs, image cache and ImageNet weights (done once, before the workers start)
import torch
N_GPU = max(1, torch.cuda.device_count())
print(N_GPU, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run([sys.executable, "-m", "fl_skin.run", "--plan", PLAN_PATH, "--data", DATA,
                "--splits", SPLITS_PATH, "--out", "/kaggle/working", "--prepare-only"],
               check=True, cwd=CODE)

In [ ]:
# 5. One worker per GPU; each takes every N_GPU-th run of the plan
os.makedirs("/kaggle/working/logs", exist_ok=True)
procs = []
for i in range(N_GPU):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONPATH=CODE)
    log = open(f"/kaggle/working/logs/{PLAN}_shard{i}.log", "a")
    procs.append(subprocess.Popen(
        [sys.executable, "-m", "fl_skin.run", "--plan", PLAN_PATH, "--data", DATA, "--splits", SPLITS_PATH,
         "--out", "/kaggle/working", "--shard", str(i), "--nshards", str(N_GPU),
         "--hours", str(HOURS), "--start", str(START)],
        cwd=CODE, env=env, stdout=log, stderr=subprocess.STDOUT))
while any(p.poll() is None for p in procs):
    time.sleep(600)
    done = len(glob.glob("/kaggle/working/runs/*/done.json"))
    print(f"{(time.time()-START)/3600:.1f} h elapsed, {done} runs finished", flush=True)
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# 6. Summary of this plan
import pandas as pd
rows = []
for r in json.load(open(PLAN_PATH)):
    d = f"/kaggle/working/runs/{r['run_id']}"
    if os.path.exists(f"{d}/done.json"):
        info = json.load(open(f"{d}/done.json")); status = "done"
    elif os.path.exists(d):
        info, status = {}, "paused (resumes next session)"
    else:
        info, status = {}, "not started"
    rows.append({"run_id": r["run_id"], "status": status, "best_val_bacc": info.get("best_val_bacc")})
summary = pd.DataFrame(rows)
summary.to_csv(f"/kaggle/working/summary_{PLAN}.csv", index=False)
print(summary["status"].value_counts().to_string()); print(summary.to_string())
for f in glob.glob("/kaggle/working/runs/calib__*/calib.json"):
    print(f, json.load(open(f)))
for i in range(N_GPU):
    print(f"--- shard {i} log tail ---"); print("".join(open(f"/kaggle/working/logs/{PLAN}_shard{i}.log").readlines()[-15:]))